# Bronze to Silver
1. Read the tables list from metadata tables
2. based on the load type of the table perform incremental logic

In [0]:
#Imports
from pyspark.sql.functions import current_timestamp, col, lit
from delta.tables import *
from datetime import datetime
from pyspark.sql.functions import max as spark_max
import json

In [0]:
catalog_name = "neo_bank"

In [0]:
tables_df = spark.read.table(f"{catalog_name}.metadata.tables")
filtered_df = tables_df.filter(col("active_flag")=="true")

In [0]:
for row in filtered_df.collect():
    table_id = row["table_id"]
    table_name = row["table_name"]
    bronze_schema = row["bronze_schema"]
    silver_schema = row["silver_schema"]


    #getting table parameters with table_id
    table_parameters_df = spark.read.table(f"{catalog_name}.metadata.table_parameters")
    table_parameters_filtered_df=table_parameters_df.filter(col("table_id")==table_id) 

    load_type_row = table_parameters_filtered_df.filter(col("parameter_name")=="load_type").select(col("parameter_value")).first()
    load_type = load_type_row[0] if load_type_row else None

    primary_key_row = table_parameters_filtered_df.filter(col("parameter_name")=="primary_key").select(col("parameter_value")).first()
    primary_key = primary_key_row[0] if primary_key_row else None

    watermark_column_row = table_parameters_filtered_df.filter(col("parameter_name")=="watermark_column").select(col("parameter_value")).first()
    watermark_column = watermark_column_row[0] if watermark_column_row else None

    #getting the last watermark value for that table with table_id
    watermark_df = spark.read.table(f"{catalog_name}.metadata.table_watermarks")
    watermark_row = watermark_df.filter(col("table_id")==table_id).select(col("last_watermark_value")).first()
    last_watermark = watermark_row[0] if watermark_row else None

    source_table = f"{catalog_name}.{bronze_schema}.{table_name}"
    target_table = f"{catalog_name}.{silver_schema}.{table_name}"

    bronze_df = spark.read.table(source_table)

    #getting the target schema
    schema_df = spark.read.table(f"{catalog_name}.metadata.table_columns")
    schema_row = schema_df.filter(col("table_id")==table_id).select(col("target_schema")).first()
    target_schema_json = schema_row["target_schema"]

    if target_schema_json is not None:
        schema_dict = json.loads(target_schema_json)
        for column_name, target_type in schema_dict.items():
            bronze_df = bronze_df.withColumn(column_name, col(column_name).cast(target_type))


    if not spark.catalog.tableExists(target_table):

        df = (
            bronze_df.withColumn("insert_timestamp",current_timestamp())
            .withColumn("update_timestamp",current_timestamp())
            )
            
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )

    else:

        if(load_type=="FULL"):
            

            df = (
                bronze_df.withColumn("insert_timestamp",current_timestamp())
                .withColumn("update_timestamp",current_timestamp())
                )
            
            (
                df.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(target_table)
            )

        if(load_type=="APPEND"):
            
            df = bronze_df.filter(col(watermark_column)>last_watermark)

            df = (
                df.withColumn("insert_timestamp",current_timestamp())
                .withColumn("update_timestamp",current_timestamp())
                )
            
            delta_table = DeltaTable.forName(spark,target_table)

            (
                delta_table.alias("t")
                .merge(
                    df.alias("s"),
                    f"t.{primary_key}=s.{primary_key}"
                )
                .whenNotMatchedInsertAll()
                .execute()
            )

        if(load_type=="MERGE"):
            
            df = bronze_df.filter(col(watermark_column)>last_watermark)

            df = (
                df.withColumn("insert_timestamp",current_timestamp())
                .withColumn("update_timestamp",current_timestamp())
                )
            
            # Exclude insert_timestamp from updates only
            update_cols = {col_name: f"s.{col_name}" for col_name in df.columns if col_name != 'insert_timestamp'}

            delta_table = DeltaTable.forName(spark,target_table)

            (
                delta_table.alias("t")
                .merge(
                    df.alias("s"),
                    f"t.{primary_key}=s.{primary_key}"
                )
                .whenMatchedUpdate(set=update_cols)
                .whenNotMatchedInsertAll()
                .execute()
            )

        
    
    #update the watermark value after successful silver ingestion
    if load_type in ['APPEND','MERGE'] and watermark_column:
        new_watermark = bronze_df.select(spark_max(col(watermark_column))).first()[0]
        watermark_table = DeltaTable.forName(spark, f"{catalog_name}.metadata.table_watermarks")
        watermark_table.update(
            condition = col("table_id") == table_id,
            set = {
                "last_watermark_value": lit(new_watermark),
                "last_updated_at": current_timestamp()
            }
        )
        


        